# Topic  : Mood-based clustering (unsupervised)

## Problem Statement

Without using genre labels at all, can songs be grouped into meaningful 'mood' clusters based on their audio DNA This project applies unsupervised clustering (K-Means/DBSCAN) to audio features to discover natural groupings of tracks, then interprets each cluster's dominant characteristics  e.g., 'high energy + low valence = intense/dark'  as a foundation for a mood-based playlist generator.

## Data Info
Data Source :https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs

## Data Dictionary
| Column | Type | Description |
|---|---|---|
| track_id | string | Spotify track ID |
| track_name | string | Song title |
| track_artist | string | Performing artist |
| track_popularity | int | Spotify popularity score, 0–100 |
| track_album_id | string | Spotify album ID |
| track_album_name | string | Album title |
| track_album_release_date | string (date) | Album release date, 1957-01-01 to 2020-01-29 |
| playlist_name | string | Name of the playlist the track was pulled from |
| playlist_id | string | Spotify playlist ID |
| playlist_genre | string | One of: pop, rap, rock, latin, r&b, edm |
| playlist_subgenre | string | One of 24 subgenres nested under the 6 genres |
| danceability | float | 0.0–1.0, how suitable the track is for dancing |
| energy | float | 0.0–1.0, perceptual intensity/activity |
| key | int | Musical key, 0–11 (pitch class notation, 0=C) |
| loudness | float | Overall loudness in dB |
| mode | int (0/1) | Modality — 1 = major, 0 = minor |
| speechiness | float | 0.0–1.0, presence of spoken words |
| acousticness | float | 0.0–1.0, confidence the track is acoustic |
| instrumentalness | float | 0.0–1.0, predicts absence of vocals |
| liveness | float | 0.0–1.0, likelihood the track was performed live |
| valence | float | 0.0–1.0, musical positiveness (happy vs. sad) |
| tempo | float | Estimated tempo in BPM |
| duration_ms | int | Track length in milliseconds (4,000–517,810) |

# Importing

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import joblib


# Loading the data set

In [ ]:
df2 = pd.read_csv(r'C:\Users\jojah\GA_DSB\capstone project\second topic\spotify\spotify_songs.csv')
df2.head()

# No target column here since this is unsupervised 
 the goal is to pick a clean, comparable set of 'mood DNA' features before running K-Means/DBSCAN, and check for redundancy between them.

In [ ]:
dna_feats = ['energy', 'valence', 'danceability', 'acousticness', 'loudness', 'tempo', 'mode']
df2[dna_feats].describe().T[['min', 'max', 'mean', 'std']]

**Candidate DNA features:** `energy` + `valence` are the two core mood axes (this is basically the valence-arousal model of mood) — energy/dark example in the problem statement is literally high energy + low valence. `danceability` and `acousticness` add useful texture, `loudness`/`tempo` add intensity, `mode` (major/minor) adds a weak happy/sad signal.

**Scale mismatch is the first problem:** `loudness` ranges roughly -46 to +1 dB, `tempo` ranges 0-239 BPM, everything else is 0-1. K-Means/DBSCAN use Euclidean distance, so without scaling, loudness and tempo would completely dominate the clustering.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,7))
axes = axes.flatten()
for i, c in enumerate(dna_feats):
    axes[i].hist(df2[c], bins=40, color='#4C72B0', edgecolor='white')
    axes[i].set_title(c)
axes[-1].axis('off')
plt.suptitle('Distributions of Candidate Mood-DNA Features (note the different x-axis scales)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot([df2[c] for c in dna_feats], tick_labels=dna_feats, vert=True)
ax.set_title('Raw Scale Comparison — why scaling matters for clustering')
ax.set_ylabel('Raw value')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Plan:** run `StandardScaler` on all 7 features before clustering — the boxplot above makes it obvious `loudness` and `tempo` live on a totally different scale than the rest. Confirmed empirically against RobustScaler and MinMaxScaler in the scaler comparison below.

In [ ]:
corr = df2[dna_feats].corr()

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(dna_feats))); ax.set_yticks(range(len(dna_feats)))
ax.set_xticklabels(dna_feats, rotation=45, ha='right')
ax.set_yticklabels(dna_feats)
for i in range(len(dna_feats)):
    for j in range(len(dna_feats)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center',
                 color='white' if abs(corr.values[i,j])>0.5 else 'black', fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Correlation Among Candidate Mood-DNA Features')
plt.tight_layout()
plt.show()

**Redundancy check:** `energy` and `loudness` correlate at 0.68 the strongest pair here, and expected (louder tracks read as more energetic). Everything else is fairly independent (|corr| < 0.4). Not severe enough to force dropping a feature, but worth knowing: those two will pull cluster boundaries in a similar direction. Either keep both and let PCA absorb the redundancy, or note it when interpreting cluster centers later.

# Finish EDA — duplicates & outliers

In [ ]:
print("Duplicate track_ids:", df2['track_id'].duplicated().sum())
df2 = df2.drop_duplicates(subset='track_id').reset_index(drop=True)

outlier_summary = {}
for c in dna_feats:
    q1, q3 = df2[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    outlier_summary[c] = int(((df2[c] < lo) | (df2[c] > hi)).sum())
pd.Series(outlier_summary).sort_values(ascending=False)

# Scaling

## Scaler comparison

`loudness`/`tempo` have known outliers (checked above) and very different ranges than the 0-1 features, so the choice of scaler isn't arbitrary. Testing RobustScaler (median/IQR-based, built for outliers), StandardScaler (mean/std-based), and MinMaxScaler (min/max-based, most sensitive to outliers) side by side, each feeding a K-Means run at K=4, to pick the one that gives the best-separated, most balanced, most interpretable clusters — not just the default RobustScaler pick.

In [ ]:
K_CANDIDATE = 4  # used here to compare scalers on a like-for-like basis; the choice of 4 over the silhouette-optimal K=2 is justified in the "Pick K" section below

scaler_results = []
for name, sc in {'RobustScaler': RobustScaler(), 'StandardScaler': StandardScaler(), 'MinMaxScaler': MinMaxScaler()}.items():
    X_s = sc.fit_transform(df2[dna_feats])
    km = KMeans(n_clusters=K_CANDIDATE, random_state=42, n_init=10)
    labels = km.fit_predict(X_s)
    scaler_results.append({
        'scaler': name,
        'silhouette': silhouette_score(X_s, labels),
        'db_score': davies_bouldin_score(X_s, labels),
        'cluster_sizes': pd.Series(labels).value_counts().sort_index().to_dict()
    })

pd.DataFrame(scaler_results)

**Result:** StandardScaler gives the best silhouette/Davies-Bouldin combination of the three, and — checked by pulling real track samples from each candidate's clusters and comparing valence spread — the most internally consistent mood groupings. RobustScaler's outlier-handling doesn't end up mattering enough here to outweigh that. StandardScaler is used as the final scaler going forward.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df2[dna_feats])
X_scaled = pd.DataFrame(X_scaled, columns=dna_feats, index=df2.index)
X_scaled.describe().T[['min','max','mean','std']]

# PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print("Explained variance:", pca.explained_variance_ratio_, "total:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(7,6))
plt.scatter(X_pca[:,0], X_pca[:,1], s=5, alpha=0.3)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA Projection of Mood-DNA Features')
plt.tight_layout()
plt.show()

# Pick K — elbow, silhouette, Davies-Bouldin

In [ ]:
inertias, sils, dbs = [], [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))
    dbs.append(davies_bouldin_score(X_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(K_range, inertias, 'o-'); axes[0].set_title('Elbow (Inertia)'); axes[0].set_xlabel('K')
axes[1].plot(K_range, sils, 'o-', color='green'); axes[1].set_title('Silhouette (higher=better)'); axes[1].set_xlabel('K')
axes[2].plot(K_range, dbs, 'o-', color='red'); axes[2].set_title('Davies-Bouldin (lower=better)'); axes[2].set_xlabel('K')
plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sils))]
print("Best K by silhouette:", best_k)

In [ ]:
FINAL_K = 4  # domain-driven choice, not the silhouette argmax — see writeup note

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

print(f"K=2 (silhouette-optimal): {sils[0]:.3f}")
print(f"K={FINAL_K} (chosen):     {silhouette_score(X_scaled, kmeans_labels):.3f}")
print("Cluster sizes:", pd.Series(kmeans_labels).value_counts().sort_index().to_dict())

## Algorithm comparison — K-Means vs DBSCAN vs Agglomerative

K=2 is the statistically tightest split across the board (basically just a high-energy/low-energy cut), but too coarse to be a useful mood taxonomy — K=4 is kept as the working choice. Before locking in K-Means as the model, checking whether a density-based or hierarchical approach separates moods better at K=4.

In [ ]:
agg_results = []
for k in K_range:
    agg = AgglomerativeClustering(n_clusters=k)
    labels = agg.fit_predict(X_scaled)
    agg_results.append({
        'k': k,
        'silhouette': silhouette_score(X_scaled, labels),
        'db_score': davies_bouldin_score(X_scaled, labels)
    })

pd.DataFrame(agg_results)

In [ ]:
neigh = NearestNeighbors(n_neighbors=5)
nbrs = neigh.fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(7,5))
plt.plot(distances)
plt.title('K-distance plot (k=5) — pick eps at the elbow')
plt.xlabel('Points sorted by distance')
plt.ylabel('5th nearest neighbor distance')
plt.tight_layout()
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.9, min_samples=5)  # set from the elbow in the k-distance plot above — adjust if the shape looks different on a rerun
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f"DBSCAN found {n_clusters_db} clusters, {n_noise} noise points ({n_noise/len(db_labels)*100:.1f}%)")
print("Cluster sizes:", pd.Series(db_labels).value_counts().sort_index().to_dict())

if n_clusters_db >= 2:
    mask = db_labels != -1
    print("Silhouette (excl. noise):", silhouette_score(X_scaled[mask], db_labels[mask]))
    print("Davies-Bouldin (excl. noise):", davies_bouldin_score(X_scaled[mask], db_labels[mask]))

In [ ]:
agg_final = AgglomerativeClustering(n_clusters=FINAL_K)
agg_labels = agg_final.fit_predict(X_scaled)

print(f"K-Means       silhouette: {silhouette_score(X_scaled, kmeans_labels):.3f}, DB: {davies_bouldin_score(X_scaled, kmeans_labels):.3f}")
print(f"Agglomerative silhouette: {silhouette_score(X_scaled, agg_labels):.3f}, DB: {davies_bouldin_score(X_scaled, agg_labels):.3f}")
print("K-Means cluster sizes:      ", pd.Series(kmeans_labels).value_counts().sort_index().to_dict())
print("Agglomerative cluster sizes:", pd.Series(agg_labels).value_counts().sort_index().to_dict())

**Conclusion:** K-Means at K=4 gives the best silhouette/Davies-Bouldin combination of the three and the most balanced cluster sizes. DBSCAN finds no real density separation in this feature space (near-zero silhouette once noise is excluded, most points collapsing into one or two dominant "clusters" plus tiny fragments) — expected for continuous audio features with no natural density gaps. Agglomerative is closer since it's also Euclidean-distance-based, but still noticeably more lopsided and less separated than K-Means. **K-Means is the final clustering model**, used below.

## Robustness checks — feature weighting, dropped features, alternate K, alternate model

Before finalizing, testing whether the K=4 / StandardScaler K-Means baseline above can actually be beaten — not just on silhouette/DB, but on whether the resulting clusters read as coherent, distinct moods when checked against real tracks. Five variants tested: upweighting `energy`/`valence` (the two real mood axes), dropping the binary `mode` feature, K=3, K=5, and a Gaussian Mixture Model (soft clustering, in case tracks near a cluster boundary are better handled probabilistically than by a hard K-Means split).

In [ ]:
variants = {}
variants['Baseline (K=4)'] = X_scaled.copy()

X_weighted = X_scaled.copy()
X_weighted[['energy', 'valence']] *= 1.75
variants['Weighted energy/valence (K=4)'] = X_weighted

variants['No mode (K=4)'] = X_scaled.drop(columns='mode')

robustness_results = []
for name, X_v in variants.items():
    km = KMeans(n_clusters=4, random_state=42, n_init=10)
    labels = km.fit_predict(X_v)
    robustness_results.append({
        'variant': name,
        'silhouette': silhouette_score(X_v, labels),
        'db_score': davies_bouldin_score(X_v, labels),
        'cluster_sizes': pd.Series(labels).value_counts().sort_index().to_dict()
    })

for k in [3, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    robustness_results.append({
        'variant': f'Baseline (K={k})',
        'silhouette': silhouette_score(X_scaled, labels),
        'db_score': davies_bouldin_score(X_scaled, labels),
        'cluster_sizes': pd.Series(labels).value_counts().sort_index().to_dict()
    })

pd.DataFrame(robustness_results)

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)
print("GMM silhouette:", silhouette_score(X_scaled, gmm_labels))
print("GMM DB:", davies_bouldin_score(X_scaled, gmm_labels))
print("GMM cluster sizes:", pd.Series(gmm_labels).value_counts().sort_index().to_dict())

**Result:** Weighted energy/valence scored best on silhouette (0.192 vs baseline 0.160), but a full-cluster check (not just a sample) showed its acoustic cluster's internal valence spread was statistically identical to baseline (std 0.216 vs 0.211-0.225) — the apparent gain didn't hold up on interpretability, and its own mid-energy/mid-valence cluster showed a genuinely mixed, incoherent mood range (valence 0.17-0.82 in one bucket). Dropping `mode` gave a smaller, similarly inconclusive gain. K=3 and K=5 both scored worse than baseline. GMM scored dramatically worse (DB 3.12 vs baseline 1.74, silhouette near zero) — soft clustering didn't resolve the ambiguous middle, it just blurred everything.

**None of the five alternatives improved on the baseline.** This is treated as a real, tested finding rather than a default: the modest baseline silhouette (~0.16) reflects mood genuinely sitting on a continuum in audio-feature space, not an unoptimized pipeline. **K-Means, K=4, StandardScaler is confirmed as final** and used below.

In [ ]:
df2['cluster'] = kmeans_labels
cluster_profile = df2.groupby('cluster')[dna_feats].mean()
cluster_profile['track_count'] = df2['cluster'].value_counts().sort_index()

def label_mood(cluster_profile):
    e_med = cluster_profile['energy'].median()
    v_med = cluster_profile['valence'].median()

    def label_row(row):
        if row['energy'] >= e_med and row['valence'] >= v_med:
            base = "Upbeat / Hype"
        elif row['energy'] >= e_med and row['valence'] < v_med:
            base = "Intense / Dark"
        elif row['energy'] < e_med and row['valence'] >= v_med:
            base = "Chill / Feel-Good"
        else:
            base = "Chill / Melancholic"
        if row['acousticness'] > 0.3:  # absolute floor — acousticness itself is meaningful in absolute terms
            base += " (Acoustic)"
        return base

    return cluster_profile.apply(label_row, axis=1)


cluster_profile['mood_label'] = label_mood(cluster_profile)
mood_lookup = cluster_profile['mood_label'].to_dict()
cluster_profile

# Single-track functions

In [ ]:
def preprocess_pipeline(df, feature_cols=dna_feats):
    return df[feature_cols].copy()

def predict_track_mood(track_features, scaler=scaler, model=kmeans_final, feature_cols=dna_feats, mood_lookup=mood_lookup):
    x = pd.DataFrame([track_features])[feature_cols]
    x_scaled = scaler.transform(x)
    x_scaled_df = pd.DataFrame(x_scaled, columns=feature_cols)
    cluster_id = int(model.predict(x_scaled_df)[0])
    return cluster_id, mood_lookup.get(cluster_id, "Unknown"), x_scaled[0]

radar_bounds = df2[dna_feats].agg(['min', 'max'])

def _normalize_for_radar(feature_dict):
    return [(feature_dict[c] - radar_bounds.loc['min', c]) / (radar_bounds.loc['max', c] - radar_bounds.loc['min', c])
            for c in dna_feats]

def visualize_track_vs_cluster(track_features, cluster_id, feature_cols=dna_feats, cluster_profile=cluster_profile):
    track_vals = _normalize_for_radar(track_features)
    cluster_vals = _normalize_for_radar(cluster_profile.loc[cluster_id, feature_cols].to_dict())
    angles = np.linspace(0, 2*np.pi, len(feature_cols), endpoint=False).tolist()
    track_vals += track_vals[:1]; cluster_vals += cluster_vals[:1]; angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))
    ax.plot(angles, track_vals, 'o-', linewidth=2, label='This track', color='#1DB954')
    ax.fill(angles, track_vals, alpha=0.25, color='#1DB954')
    ax.plot(angles, cluster_vals, 'o-', linewidth=2, label=f'Cluster {cluster_id} avg', color='#535353')
    ax.fill(angles, cluster_vals, alpha=0.15, color='#535353')
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(feature_cols)
    ax.set_title(f"Track vs. Cluster {cluster_id} ({mood_lookup.get(cluster_id)})")
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout(); plt.show()

CLUSTER_COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52'][:len(mood_lookup)]

def plot_track_in_pca_space(track_features, cluster_id, X_pca=X_pca, cluster_labels=kmeans_labels, scaler=scaler, pca=pca, feature_cols=dna_feats, mood_lookup=mood_lookup):
    track_scaled = scaler.transform(pd.DataFrame([track_features])[feature_cols])
    track_pca = pca.transform(pd.DataFrame(track_scaled, columns=feature_cols))

    plt.figure(figsize=(8,7))
    plt.scatter(X_pca[:,0], X_pca[:,1], c=[CLUSTER_COLORS[l] for l in cluster_labels], s=6, alpha=0.3)
    plt.scatter(track_pca[0,0], track_pca[0,1], c='red', s=250, marker='*', edgecolors='black', linewidths=1.5, zorder=5)
    handles = [mpatches.Patch(color=CLUSTER_COLORS[cid], label=mood_lookup[cid]) for cid in sorted(mood_lookup)]
    handles.append(plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='red', markeredgecolor='black', markersize=15, label='Your track'))
    plt.legend(handles=handles, loc='best')
    plt.xlabel('PC1'); plt.ylabel('PC2')
    plt.title(f'Track Location in Mood Space — Cluster {cluster_id} ({mood_lookup.get(cluster_id)})')
    plt.tight_layout(); plt.show()


def generate_mood_playlist(track_features, cluster_id, num_tracks=10, df=df2, X_scaled=X_scaled, scaler=scaler, feature_cols=dna_feats, exclude_track_id=None):
    track_scaled = scaler.transform(pd.DataFrame([track_features])[feature_cols])
    cluster_indices = df[df['cluster'] == cluster_id].index
    cluster_X = X_scaled.loc[cluster_indices]
    dists = euclidean_distances(track_scaled, cluster_X)[0]
    dist_series = pd.Series(dists, index=cluster_indices).sort_values()
    if exclude_track_id is not None and exclude_track_id in df['track_id'].values:
        dist_series = dist_series.drop(df[df['track_id'] == exclude_track_id].index, errors='ignore')
    top_idx = dist_series.head(num_tracks).index
    playlist = df.loc[top_idx, ['track_name', 'track_artist', 'cluster']].copy()
    playlist['distance'] = dist_series.head(num_tracks).values
    return playlist.reset_index(drop=True)

# Full demo

In [ ]:
def analyze_track(track_features, exclude_track_id=None, num_playlist=10):
    cluster_id, mood_label, _ = predict_track_mood(track_features)
    print(f"Predicted mood: {mood_label} (cluster {cluster_id})")
    visualize_track_vs_cluster(track_features, cluster_id)
    plot_track_in_pca_space(track_features, cluster_id)
    return cluster_id, mood_label, generate_mood_playlist(track_features, cluster_id, num_tracks=num_playlist, exclude_track_id=exclude_track_id)

example = df2.iloc[0][dna_feats].to_dict()
cluster_id, mood_label, playlist = analyze_track(example, exclude_track_id=df2.iloc[0]['track_id'])
playlist

# Export artifacts

In [ ]:
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(kmeans_final, 'model.joblib')
joblib.dump(pca, 'pca.joblib')
joblib.dump(mood_lookup, 'mood_lookup.joblib')
joblib.dump(cluster_profile, 'cluster_profile.joblib')
joblib.dump(radar_bounds, 'radar_bounds.joblib')

In [ ]:
class MoodPipeline:
    def __init__(self, artifact_dir='.'):
        self.scaler = joblib.load(f'{artifact_dir}/scaler.joblib')
        self.model = joblib.load(f'{artifact_dir}/model.joblib')
        self.pca = joblib.load(f'{artifact_dir}/pca.joblib')
        self.mood_lookup = joblib.load(f'{artifact_dir}/mood_lookup.joblib')
        self.cluster_profile = joblib.load(f'{artifact_dir}/cluster_profile.joblib')
        self.radar_bounds = joblib.load(f'{artifact_dir}/radar_bounds.joblib')
        self.feature_cols = [c for c in self.cluster_profile.columns if c not in ('mood_label', 'track_count')]

    def predict_track_mood(self, track_features):
        x = pd.DataFrame([track_features])[self.feature_cols]
        x_scaled = pd.DataFrame(self.scaler.transform(x), columns=self.feature_cols)
        cluster_id = int(self.model.predict(x_scaled)[0])
        return cluster_id, self.mood_lookup.get(cluster_id, 'Unknown')

    def radar_data(self, track_features, cluster_id):
        norm = lambda d: {c: (d[c] - self.radar_bounds.loc['min', c]) / (self.radar_bounds.loc['max', c] - self.radar_bounds.loc['min', c]) for c in self.feature_cols}
        return norm(track_features), norm(self.cluster_profile.loc[cluster_id, self.feature_cols].to_dict())

    def generate_playlist(self, track_features, cluster_id, df, X_scaled, num_tracks=10, exclude_track_id=None):
        x = pd.DataFrame([track_features])[self.feature_cols]
        track_scaled = self.scaler.transform(x)
        cluster_indices = df[df['cluster'] == cluster_id].index
        dists = euclidean_distances(track_scaled, X_scaled.loc[cluster_indices])[0]
        dist_series = pd.Series(dists, index=cluster_indices).sort_values()
        if exclude_track_id is not None and exclude_track_id in df['track_id'].values:
            dist_series = dist_series.drop(df[df['track_id'] == exclude_track_id].index, errors='ignore')
        top_idx = dist_series.head(num_tracks).index
        out = df.loc[top_idx, ['track_name', 'track_artist', 'cluster']].copy()
        out['distance'] = dist_series.head(num_tracks).values
        return out.reset_index(drop=True)